# FEATURE IMPORTANCE

### Return Variable 

In [20]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

# ----------------------------------------------------------------
# ------- Load the model 
# ----------------------------------------------------------------

MODEL_PATH = Path("D://Joseph/Projects/kg-house-price-prediction/artifacts/trained_models/xgboost_regression.joblib")
model = joblib.load(MODEL_PATH)
print(model)

# --- 1. Split the pipeline into preprocessing steps + regressor ---
preprocessing = model[:-1]                 # everything before the regressor
regressor = model.named_steps['regressor'] # the XGBRegressor itself


print(f"Print preprocessing {preprocessing}")

# ----------------------------------------------------------------
# --- Load your data (adjust path/loading as needed) ---
# ----------------------------------------------------------------

X_raw = pd.read_csv("D://Joseph/Projects/kg-house-price-prediction/data/raw/test.csv")


# --- 2. Transform data through the preprocessing steps ---
X_transformed = preprocessing.transform(X_raw)

# Convert sparse -> dense before wrapping in a DataFrame
if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = pd.DataFrame(X_transformed, columns=feature_names)

print(X_transformed.shape)

print("Print Data")
print(X_transformed)
print(type(X_transformed))
print(X_transformed.shape)

# Try to recover proper feature names after transformation
try:
    feature_names = preprocessing.get_feature_names_out()
except AttributeError:
    # Fall back if get_feature_names_out isn't implemented on a custom step
    feature_names = [f"feat_{i}" for i in range(X_transformed.shape[1])]

X_transformed = pd.DataFrame(X_transformed, columns=feature_names)

print(f"\nTransformed shape: {X_transformed.shape}")
print(f"Feature names: {list(feature_names)}")


Pipeline(steps=[('feature_engineer', FeatureEngineer()),
                ('dropper', MissingRatioDropper(min_ratio=0.0)),
                ('preprocessor', Preprocessor()),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_stopping_rounds=None,
                              enable_categorical=False, eval_metric='rm...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth

(1459, 313)
Print Data
        feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0     1.717115 -0.866764  0.430047  0.086693 -0.820445  0.372217 -0.325016   
1     1.719467 -0.866764  0.474583  0.332630 -0.088934  0.372217 -0.422856   
2     1.721819  0.074110  0.162831  0.291997 -0.820445 -0.524174  0.849062   
3     1.724171  0.074110  0.340975 -0.066170 -0.088934  0.372217  0.881675   
4     1.726523  1.485421 -1.217789 -0.528570  1.374088 -0.524174  0.685996   
...        ...       ...       ...       ...       ...       ...       ...   
1454  5.136787  2.426296 -2.197583 -0.813932 -1.551955  1.268609 -0.031496   
1455  5.139139  2.426296 -2.197583 -0.817837 -1.551955 -0.524174 -0.031496   
1456  5.141491 -0.866764  3.992936  0.865697 -0.820445  1.268609 -0.357629   
1457  5.143843  0.662156 -0.371603 -0.023119 -0.820445 -0.524174  0.685996   
1458  5.146195  0.074110  0.162831 -0.098807  0.642577 -0.524174  0.718609   

        feat_7    feat_8    feat_9  ... 

In [ ]:

# --- 3. Built-in XGBoost gain importance ---
gain_importance = regressor.get_booster().get_score(importance_type='gain')
# XGBoost internally labels features f0, f1, ... — map back to real names
booster_feature_map = {f"f{i}": name for i, name in enumerate(feature_names)}
gain_series = pd.Series({booster_feature_map.get(k, k): v for k, v in gain_importance.items()})
gain_series = gain_series.sort_values(ascending=False)
print("\n=== Gain importance ===")
print(gain_series)

gain_series.plot(kind='barh', figsize=(8, max(4, len(gain_series) * 0.3)))
plt.title("XGBoost Gain Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('gain_importance.png', dpi=150)
plt.close()

# --- 4. SHAP global importance ---
explainer = shap.TreeExplainer(regressor)
shap_values = explainer.shap_values(X_transformed)

shap.summary_plot(shap_values, X_transformed, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150)
plt.close()

shap.summary_plot(shap_values, X_transformed, show=False)  # beeswarm
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150)
plt.close()

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0), index=feature_names
).sort_values(ascending=False)
print("\n=== Mean |SHAP value| ===")
print(mean_abs_shap)

# --- 5. Permutation importance (needs y_raw / y_true) ---
from sklearn.inspection import permutation_importance

# Permutation importance should run on the FULL pipeline against raw X, raw y,
# so preprocessing is re-applied correctly on each shuffle
perm = permutation_importance(
    model, X_raw, y_raw, n_repeats=10, random_state=0, scoring='neg_mean_squared_error'
)
perm_series = pd.Series(perm.importances_mean, index=X_raw.columns).sort_values(ascending=False)
print("\n=== Permutation importance (raw feature space) ===")
print(perm_series)

# --- 6. SHAP interaction values (optional, can be slow) ---
# Subsample if X_transformed is large
sample = X_transformed.sample(min(500, len(X_transformed)), random_state=0)
interaction_values = explainer.shap_interaction_values(sample)
mean_abs_interactions = np.abs(interaction_values).mean(axis=0)
interaction_df = pd.DataFrame(mean_abs_interactions, index=feature_names, columns=feature_names)
np.fill_diagonal(interaction_df.values, 0)
top_pairs = interaction_df.unstack().sort_values(ascending=False).drop_duplicates().head(10)
print("\n=== Top feature interactions ===")
print(top_pairs)